In [1]:
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Load Static Features and CA Simulation
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
df_ca = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "ca_simulation_camp_fire.parquet"))

df_merged = df_static.merge(df_ca[["h3_id", "predicted_arrival_min", "is_burned_12h"]], on="h3_id")

# 2. Define Survival Targets (Duration T and Event E)
OBSERVATION_WINDOW_MIN = 720.0  # 12 hours

# If burned within observation window, duration = arrival time, event = 1
# If unburned / arrival > 720, duration = 720.0 (right-censored), event = 0
duration = np.where(
    df_merged["predicted_arrival_min"] < OBSERVATION_WINDOW_MIN,
    df_merged["predicted_arrival_min"],
    OBSERVATION_WINDOW_MIN
).astype(np.float32)

# Ensure minimum non-zero duration for log-time transforms in AFT
duration = np.maximum(duration, 1.0)
event = np.where(df_merged["predicted_arrival_min"] < OBSERVATION_WINDOW_MIN, 1, 0).astype(np.int32)

df_merged["duration_min"] = duration
df_merged["event_burned"] = event

# 3. Spatial & Meteorological Feature Engineering
# Identify ignition centroid
ignition_cells = df_merged[df_merged["predicted_arrival_min"] == 0.0]
if len(ignition_cells) > 0:
    ign_lat = ignition_cells["centroid_lat"].mean()
    ign_lon = ignition_cells["centroid_lon"].mean()
else:
    ign_lat, ign_lon = 39.8, -121.55

print(f"Ignition centroid reference: ({ign_lat:.4f}, {ign_lon:.4f})")

# Distance to ignition centroid (km) and compass direction from ignition
d_lat = (df_merged["centroid_lat"].values - ign_lat) * 111.0
d_lon = (df_merged["centroid_lon"].values - ign_lon) * 111.0 * np.cos(np.radians(ign_lat))
dist_to_ign_km = np.sqrt(d_lat**2 + d_lon**2).astype(np.float32)
angle_from_ign_rad = np.mod(np.arctan2(d_lon, d_lat) + 2 * np.pi, 2 * np.pi).astype(np.float32)

df_merged["dist_to_ignition_km"] = dist_to_ign_km
df_merged["aspect_sin"] = np.sin(df_merged["aspect_mean_rad"]).astype(np.float32)
df_merged["aspect_cos"] = np.cos(df_merged["aspect_mean_rad"]).astype(np.float32)

# Wind alignment relative to ignition vector
df_merged["wind_ign_alignment"] = np.cos(angle_from_ign_rad - np.radians(45.0)).astype(np.float32) # Dominant NE wind

# Fuel burnability indicator
df_merged["is_burnable_fuel"] = (df_merged["fuel_model"] != 99).astype(np.int8)

# Group assignment for spatial cross-validation (4 geographic quadrants)
lat_median = df_merged["centroid_lat"].median()
lon_median = df_merged["centroid_lon"].median()

quadrant_group = np.zeros(len(df_merged), dtype=np.int32)
quadrant_group[(df_merged["centroid_lat"] >= lat_median) & (df_merged["centroid_lon"] >= lon_median)] = 0  # NE
quadrant_group[(df_merged["centroid_lat"] >= lat_median) & (df_merged["centroid_lon"] < lon_median)] = 1   # NW
quadrant_group[(df_merged["centroid_lat"] < lat_median) & (df_merged["centroid_lon"] >= lon_median)] = 2   # SE
quadrant_group[(df_merged["centroid_lat"] < lat_median) & (df_merged["centroid_lon"] < lon_median)] = 3    # SW

df_merged["spatial_group"] = quadrant_group

# 4. Save Survival Training Dataset
output_survival_path = os.path.join(PROCESSED_DATA_DIR, "survival_training_data.parquet")
table = pa.Table.from_pandas(df_merged)
pq.write_table(table, output_survival_path, compression="snappy")

print("\n--- Survival Dataset Summary ---")
print(f"Total instances: {len(df_merged)}")
print(f"Burned events (E=1): {event.sum()} ({event.mean()*100:.1f} %)")
print(f"Right-censored (E=0): {(1-event).sum()} ({(1-event.mean())*100:.1f} %)")
print(f"Spatial groups distribution:\n{pd.Series(quadrant_group).value_counts().sort_index().to_dict()}")
print(f"\nSaved dataset to: {output_survival_path}")

Ignition centroid reference: (39.8273, -121.6008)

--- Survival Dataset Summary ---
Total instances: 2526
Burned events (E=1): 1981 (78.4 %)
Right-censored (E=0): 545 (21.6 %)
Spatial groups distribution:
{0: 630, 1: 633, 2: 633, 3: 630}

Saved dataset to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\survival_training_data.parquet


In [2]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import brier_score_loss

# 1. Load Training Data
data_path = os.path.join(PROCESSED_DATA_DIR, "survival_training_data.parquet")
df = pd.read_parquet(data_path)

feature_cols = [
    "elevation_mean_m",
    "slope_mean_deg",
    "aspect_sin",
    "aspect_cos",
    "fuel_model",
    "fuel_frac_grass",
    "fuel_frac_shrub",
    "fuel_frac_timber",
    "drought_category",
    "pm25_baseline_ugm3",
    "dist_to_ignition_km",
    "wind_ign_alignment",
    "temp_c_t0",
    "wind_speed_ms_t0",
    "rh_pct_t0"
]

X = df[feature_cols].copy()
duration = df["duration_min"].values
event = df["event_burned"].values
groups = df["spatial_group"].values

# 2. Train/Test Split by Spatial Groups (Train: Groups 0, 1, 2; Test: Group 3)
train_mask = groups != 3
test_mask = groups == 3

X_train, X_test = X[train_mask], X[test_mask]
dur_train, dur_test = duration[train_mask], duration[test_mask]
ev_train, ev_test = event[train_mask], event[test_mask]

print(f"Training Set: {len(X_train)} samples ({ev_train.sum()} burned, {len(X_train) - ev_train.sum()} censored)")
print(f"Testing Set:  {len(X_test)} samples ({ev_test.sum()} burned, {len(X_test) - ev_test.sum()} censored)")

# 3. Format Data for XGBoost Survival
# For AFT: bounds [y_lower, y_upper]. If event=1, [T, T]. If event=0 (censored), [T, +inf]
y_lower_train = dur_train
y_upper_train = np.where(ev_train == 1, dur_train, np.inf)

y_lower_test = dur_test
y_upper_test = np.where(ev_test == 1, dur_test, np.inf)

dtrain_aft = xgb.DMatrix(X_train)
dtrain_aft.set_float_info("label_lower_bound", y_lower_train)
dtrain_aft.set_float_info("label_upper_bound", y_upper_train)

dtest_aft = xgb.DMatrix(X_test)
dtest_aft.set_float_info("label_lower_bound", y_lower_test)
dtest_aft.set_float_info("label_upper_bound", y_upper_test)

# For Cox: positive label for uncensored, negative label for right-censored
y_cox_train = np.where(ev_train == 1, dur_train, -dur_train)
y_cox_test = np.where(ev_test == 1, dur_test, -dur_test)

dtrain_cox = xgb.DMatrix(X_train, label=y_cox_train)
dtest_cox = xgb.DMatrix(X_test, label=y_cox_test)

# 4. Train XGBoost AFT Model
params_aft = {
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "aft_loss_distribution": "normal",
    "aft_loss_distribution_scale": 1.0,
    "tree_method": "hist",
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

model_aft = xgb.train(
    params_aft,
    dtrain_aft,
    num_boost_round=200,
    evals=[(dtrain_aft, "train"), (dtest_aft, "test")],
    verbose_eval=False
)

# 5. Train XGBoost Cox Model
params_cox = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    "learning_rate": 0.05,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

model_cox = xgb.train(
    params_cox,
    dtrain_cox,
    num_boost_round=150,
    evals=[(dtrain_cox, "train"), (dtest_cox, "test")],
    verbose_eval=False
)

# 6. Evaluation: Harrell's Concordance Index (C-index)
def concordance_index(durations, predictions, events):
    """Computes Harrell's C-index for survival evaluation."""
    n = len(durations)
    concordant, permissible = 0, 0
    for i in range(n):
        for j in range(i + 1, n):
            # At least one had event, or both had events with different times
            if (events[i] == 1 and durations[i] < durations[j]) or (events[j] == 1 and durations[j] < durations[i]):
                permissible += 1
                if durations[i] < durations[j]:
                    # Lower predicted duration -> earlier event (AFT)
                    if predictions[i] < predictions[j]:
                        concordant += 1
                    elif predictions[i] == predictions[j]:
                        concordant += 0.5
                else:
                    if predictions[j] < predictions[i]:
                        concordant += 1
                    elif predictions[j] == predictions[i]:
                        concordant += 0.5
    return concordant / permissible if permissible > 0 else 0.5

# Predictions on Test Set
pred_aft_log_time = model_aft.predict(dtest_aft)
pred_aft_time = np.exp(pred_aft_log_time)  # Predicted time to burn (min)

# Cox risk scores: higher score = shorter survival
pred_cox_risk = model_cox.predict(dtest_cox)
pred_cox_surv_proxy = -pred_cox_risk  # Invert so lower value = earlier event

c_index_aft = concordance_index(dur_test, pred_aft_time, ev_test)
c_index_cox = concordance_index(dur_test, pred_cox_surv_proxy, ev_test)

# Brier Scores at 3h (180m), 6h (360m), 12h (720m) using AFT probability
# Standard logistic/normal survival function: S(t) = P(T > t)
scale = params_aft["aft_loss_distribution_scale"]
z_3h = (np.log(180.0) - pred_aft_log_time) / scale
z_6h = (np.log(360.0) - pred_aft_log_time) / scale
z_12h = (np.log(720.0) - pred_aft_log_time) / scale

prob_burn_3h = (1.0 / (1.0 + np.exp(-z_3h))).astype(np.float32)
prob_burn_6h = (1.0 / (1.0 + np.exp(-z_6h))).astype(np.float32)
prob_burn_12h = (1.0 / (1.0 + np.exp(-z_12h))).astype(np.float32)

actual_burned_3h = (dur_test <= 180.0) & (ev_test == 1)
actual_burned_6h = (dur_test <= 360.0) & (ev_test == 1)
actual_burned_12h = (dur_test <= 720.0) & (ev_test == 1)

brier_3h = brier_score_loss(actual_burned_3h, prob_burn_3h)
brier_6h = brier_score_loss(actual_burned_6h, prob_burn_6h)
brier_12h = brier_score_loss(actual_burned_12h, prob_burn_12h)

print("\n=======================================================")
print("SURVIVAL MODEL TEST SET EVALUATION")
print("=======================================================")
print(f"AFT Model Harrell's C-Index: {c_index_aft:.4f}")
print(f"Cox Model Harrell's C-Index: {c_index_cox:.4f}")
print(f"AFT Brier Score @ 3h:       {brier_3h:.4f}")
print(f"AFT Brier Score @ 6h:       {brier_6h:.4f}")
print(f"AFT Brier Score @ 12h:      {brier_12h:.4f}")
print("=======================================================")

# 7. Save Models and Feature Importance
models_dir = os.path.join(PROJECT_ROOT, "models")
reports_dir = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

model_aft.save_model(os.path.join(models_dir, "survival_aft.json"))
model_cox.save_model(os.path.join(models_dir, "survival_cox.json"))

# Feature importance table
importance_dict = model_aft.get_score(importance_type="gain")
df_importance = pd.DataFrame([
    {"feature": k, "gain": v} for k, v in importance_dict.items()
]).sort_values("gain", ascending=False)

df_importance.to_csv(os.path.join(models_dir, "feature_importance.csv"), index=False)

metrics = {
    "c_index_aft": float(c_index_aft),
    "c_index_cox": float(c_index_cox),
    "brier_3h": float(brier_3h),
    "brier_6h": float(brier_6h),
    "brier_12h": float(brier_12h)
}

with open(os.path.join(reports_dir, "survival_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("\nTop 5 Most Informative Features (AFT Gain):")
print(df_importance.head(5))

ModuleNotFoundError: No module named 'xgboost'

In [3]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor

# 1. Load Training Data
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
data_path = os.path.join(PROCESSED_DATA_DIR, "survival_training_data.parquet")
df = pd.read_parquet(data_path)

feature_cols = [
    "elevation_mean_m",
    "slope_mean_deg",
    "aspect_sin",
    "aspect_cos",
    "fuel_model",
    "fuel_frac_grass",
    "fuel_frac_shrub",
    "fuel_frac_timber",
    "drought_category",
    "pm25_baseline_ugm3",
    "dist_to_ignition_km",
    "wind_ign_alignment",
    "temp_c_t0",
    "wind_speed_ms_t0",
    "rh_pct_t0"
]

X = df[feature_cols].copy()
duration = df["duration_min"].values
event = df["event_burned"].values
groups = df["spatial_group"].values

# 2. Train/Test Split by Spatial Groups (Train: Groups 0, 1, 2; Test: Group 3)
train_mask = groups != 3
test_mask = groups == 3

X_train, X_test = X[train_mask], X[test_mask]
dur_train, dur_test = duration[train_mask], duration[test_mask]
ev_train, ev_test = event[train_mask], event[test_mask]

print(f"Training Set: {len(X_train)} samples ({ev_train.sum()} burned, {len(X_train) - ev_train.sum()} censored)")
print(f"Testing Set:  {len(X_test)} samples ({ev_test.sum()} burned, {len(X_test) - ev_test.sum()} censored)")

# 3. Check for XGBoost or train with HistGradientBoosting Survival / Log-Time Estimator
has_xgboost = False
try:
    import xgboost as xgb
    has_xgboost = True
except ImportError:
    pass

if has_xgboost:
    print("\nUsing XGBoost Survival Engine (AFT & Cox)...")
    y_lower_train = dur_train
    y_upper_train = np.where(ev_train == 1, dur_train, np.inf)
    y_lower_test = dur_test
    y_upper_test = np.where(ev_test == 1, dur_test, np.inf)

    dtrain_aft = xgb.DMatrix(X_train)
    dtrain_aft.set_float_info("label_lower_bound", y_lower_train)
    dtrain_aft.set_float_info("label_upper_bound", y_upper_train)

    dtest_aft = xgb.DMatrix(X_test)
    dtest_aft.set_float_info("label_lower_bound", y_lower_test)
    dtest_aft.set_float_info("label_upper_bound", y_upper_test)

    params_aft = {
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 1.0,
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 5,
        "seed": 42
    }
    model_aft = xgb.train(params_aft, dtrain_aft, num_boost_round=200)
    pred_aft_log_time = model_aft.predict(dtest_aft)
    pred_aft_time = np.exp(pred_aft_log_time)
else:
    print("\nUsing Scikit-Learn HistGradientBoosting Log-Duration Survival Estimator...")
    # Log survival duration target with sample weights for right-censoring
    log_dur_train = np.log(dur_train)
    # Give higher weight to confirmed uncensored events
    sample_weights = np.where(ev_train == 1, 1.0, 0.5)

    model_aft = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    )
    model_aft.fit(X_train, log_dur_train, sample_weight=sample_weights)
    pred_aft_log_time = model_aft.predict(X_test)
    pred_aft_time = np.exp(pred_aft_log_time)

# 4. Evaluation: Harrell's Concordance Index (C-index)
def concordance_index(durations, predictions, events):
    """Harrell's C-index implementation."""
    n = len(durations)
    concordant, permissible = 0, 0
    for i in range(n):
        for j in range(i + 1, n):
            if (events[i] == 1 and durations[i] < durations[j]) or (events[j] == 1 and durations[j] < durations[i]):
                permissible += 1
                if durations[i] < durations[j]:
                    if predictions[i] < predictions[j]:
                        concordant += 1
                    elif predictions[i] == predictions[j]:
                        concordant += 0.5
                else:
                    if predictions[j] < predictions[i]:
                        concordant += 1
                    elif predictions[j] == predictions[i]:
                        concordant += 0.5
    return concordant / permissible if permissible > 0 else 0.5

c_index = concordance_index(dur_test, pred_aft_time, ev_test)

# Brier Scores at 3h (180m), 6h (360m), 12h (720m)
# Survival probability P(T <= t) estimated via logistic cumulative CDF over residuals
residual_scale = 0.8
z_3h = (np.log(180.0) - pred_aft_log_time) / residual_scale
z_6h = (np.log(360.0) - pred_aft_log_time) / residual_scale
z_12h = (np.log(720.0) - pred_aft_log_time) / residual_scale

prob_burn_3h = (1.0 / (1.0 + np.exp(-z_3h))).astype(np.float32)
prob_burn_6h = (1.0 / (1.0 + np.exp(-z_6h))).astype(np.float32)
prob_burn_12h = (1.0 / (1.0 + np.exp(-z_12h))).astype(np.float32)

actual_burned_3h = (dur_test <= 180.0) & (ev_test == 1)
actual_burned_6h = (dur_test <= 360.0) & (ev_test == 1)
actual_burned_12h = (dur_test <= 720.0) & (ev_test == 1)

brier_3h = brier_score_loss(actual_burned_3h, prob_burn_3h)
brier_6h = brier_score_loss(actual_burned_6h, prob_burn_6h)
brier_12h = brier_score_loss(actual_burned_12h, prob_burn_12h)

print("\n=======================================================")
print("SURVIVAL MODEL TEST SET EVALUATION")
print("=======================================================")
print(f"Harrell's Concordance Index (C-Index): {c_index:.4f}")
print(f"Brier Score @ 3h  (180 min):          {brier_3h:.4f}")
print(f"Brier Score @ 6h  (360 min):          {brier_6h:.4f}")
print(f"Brier Score @ 12h (720 min):          {brier_12h:.4f}")
print("=======================================================")

# 5. Generate Full-Grid Predictions across All 2,526 Cells
if has_xgboost:
    dall = xgb.DMatrix(X)
    all_pred_log_time = model_aft.predict(dall)
else:
    all_pred_log_time = model_aft.predict(X)

all_pred_time = np.exp(all_pred_log_time)

# Compute cumulative burn probabilities for all forecast horizons
z_all_3h = (np.log(180.0) - all_pred_log_time) / residual_scale
z_all_6h = (np.log(360.0) - all_pred_log_time) / residual_scale
z_all_12h = (np.log(720.0) - all_pred_log_time) / residual_scale

df["predicted_burn_time_min"] = all_pred_time.astype(np.float32)
df["burn_prob_3h"] = (1.0 / (1.0 + np.exp(-z_all_3h))).astype(np.float32)
df["burn_prob_6h"] = (1.0 / (1.0 + np.exp(-z_all_6h))).astype(np.float32)
df["burn_prob_12h"] = (1.0 / (1.0 + np.exp(-z_all_12h))).astype(np.float32)

# Save full survival predictions table
models_dir = os.path.join(PROJECT_ROOT, "models")
reports_dir = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

output_pred_path = os.path.join(PROCESSED_DATA_DIR, "survival_predictions_grid.parquet")
df[["h3_id", "predicted_burn_time_min", "burn_prob_3h", "burn_prob_6h", "burn_prob_12h"]].to_parquet(
    output_pred_path, compression="snappy"
)

metrics = {
    "c_index": float(c_index),
    "brier_3h": float(brier_3h),
    "brier_6h": float(brier_6h),
    "brier_12h": float(brier_12h)
}

with open(os.path.join(reports_dir, "survival_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nSuccessfully generated full grid survival probabilities: {output_pred_path}")
print(df[["h3_id", "predicted_burn_time_min", "burn_prob_3h", "burn_prob_6h", "burn_prob_12h"]].head(5))

Training Set: 1896 samples (1351 burned, 545 censored)
Testing Set:  630 samples (630 burned, 0 censored)

Using Scikit-Learn HistGradientBoosting Log-Duration Survival Estimator...

SURVIVAL MODEL TEST SET EVALUATION
Harrell's Concordance Index (C-Index): 0.7124
Brier Score @ 3h  (180 min):          0.1733
Brier Score @ 6h  (360 min):          0.1791
Brier Score @ 12h (720 min):          0.0371

Successfully generated full grid survival probabilities: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\survival_predictions_grid.parquet
                h3_id  predicted_burn_time_min  burn_prob_3h  burn_prob_6h  \
0  613194612432961535               451.340149      0.240657      0.429805   
1  613194612441350143               611.727966      0.178116      0.340127   
2  613194612456030207               639.293457      0.170193      0.327872   
3  613194612458127359               531.702148      0.205232      0.380489   
4  613194612460224511               616.796814      0.176611